# Custom CNN training
This notebook is the executable end-to-end training/evaluation module. It uses only the training split for augmentation and class weights; validation drives callbacks; test is evaluated once after model selection.


In [ ]:
import json, time, csv
from pathlib import Path
import numpy as np, pandas as pd, tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report,confusion_matrix
import matplotlib.pyplot as plt, seaborn as sns
ROOT=Path.cwd().parent if Path.cwd().name=="notebooks" else Path.cwd(); DATASET=ROOT/"PlantVillage"; OUT=ROOT/"results/custom_cnn"; OUT.mkdir(parents=True,exist_ok=True); MODEL=ROOT/"models"; MODEL.mkdir(exist_ok=True)
IMAGE_SIZE=(224,224); BATCH_SIZE=32; EPOCHS=30; LEARNING_RATE=.001; RANDOM_SEED=42; USE_CLASS_WEIGHTS=True
tf.keras.utils.set_random_seed(RANDOM_SEED)
meta=pd.read_csv(ROOT/"results/preprocessing/dataset_split.csv"); classes=sorted(meta.class_name.unique()); NUM_CLASSES=len(classes)
aug=tf.keras.Sequential([tf.keras.layers.RandomFlip("horizontal"),tf.keras.layers.RandomRotation(.05),tf.keras.layers.RandomZoom(.1),tf.keras.layers.RandomBrightness(.08)])
def ds(frame,training=False):
 x=tf.data.Dataset.from_tensor_slices((frame.filepath.values,frame.class_index.values))
 def load(p,y):
  im=tf.io.read_file(p); im=tf.image.decode_image(im,channels=3,expand_animations=False); im=tf.image.resize(im,IMAGE_SIZE)/255.0
  if training: im=aug(im,training=True)
  return im,y
 return x.map(load,num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
train,valid,test=[ds(meta[meta.split==s],s=="train") for s in ["train","validation","test"]]
model=tf.keras.Sequential([tf.keras.Input((*IMAGE_SIZE,3)),tf.keras.layers.Conv2D(32,3,padding="same",activation="relu"),tf.keras.layers.BatchNormalization(),tf.keras.layers.MaxPooling2D(),tf.keras.layers.Conv2D(64,3,padding="same",activation="relu"),tf.keras.layers.BatchNormalization(),tf.keras.layers.MaxPooling2D(),tf.keras.layers.Conv2D(128,3,padding="same",activation="relu"),tf.keras.layers.BatchNormalization(),tf.keras.layers.MaxPooling2D(),tf.keras.layers.Conv2D(256,3,padding="same",activation="relu"),tf.keras.layers.BatchNormalization(),tf.keras.layers.GlobalAveragePooling2D(),tf.keras.layers.Dense(128,activation="relu"),tf.keras.layers.Dropout(.5),tf.keras.layers.Dense(NUM_CLASSES,activation="softmax")])
model.compile(tf.keras.optimizers.Adam(LEARNING_RATE),loss=tf.keras.losses.SparseCategoricalCrossentropy(),metrics=["accuracy"])
y=meta.loc[meta.split=="train","class_index"]; cw=pd.read_csv(ROOT/"results/preprocessing/class_weights.csv").weight.to_numpy(); class_weights=dict(enumerate(cw))
cb=[tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=5,restore_best_weights=True),tf.keras.callbacks.ModelCheckpoint(MODEL/"custom_cnn_best.keras",monitor="val_loss",save_best_only=True),tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss",patience=2)]
t0=time.time(); h=model.fit(train,validation_data=valid,epochs=EPOCHS,class_weight=class_weights if USE_CLASS_WEIGHTS else None,callbacks=cb); elapsed=time.time()-t0
pd.DataFrame(h.history).to_csv(OUT/"training_history.csv",index=False)
for key,title,file in [("accuracy","Accuracy","accuracy_curve.png"),("loss","Loss","loss_curve.png")]:
 plt.figure(); plt.plot(h.history[key],label="train"); plt.plot(h.history["val_"+key],label="validation"); plt.xlabel("Epoch"); plt.ylabel(title); plt.legend(); plt.tight_layout(); plt.savefig(OUT/file); plt.close()
pr=model.predict(test); yp=pr.argmax(1); yt=meta.loc[meta.split=="test","class_index"].to_numpy(); rep=classification_report(yt,yp,target_names=classes,output_dict=True); pd.DataFrame(rep).T.to_csv(OUT/"classification_report.csv")
cm=confusion_matrix(yt,yp); plt.figure(figsize=(12,10)); sns.heatmap(cm,annot=False,xticklabels=classes,yticklabels=classes); plt.tight_layout(); plt.savefig(OUT/"confusion_matrix.png"); plt.close()
metrics={"Model":"Custom CNN","Test Accuracy":rep["accuracy"],"Macro Precision":rep["macro avg"]["precision"],"Macro Recall":rep["macro avg"]["recall"],"Macro F1":rep["macro avg"]["f1-score"],"Weighted Precision":rep["weighted avg"]["precision"],"Weighted Recall":rep["weighted avg"]["recall"],"Weighted F1":rep["weighted avg"]["f1-score"]}; pd.DataFrame([metrics]).to_csv(OUT/"final_metrics.csv",index=False)
pd.DataFrame([{ "total_parameters":model.count_params(),"trainable_parameters":sum(np.prod(v.shape) for v in model.trainable_weights),"non_trainable_parameters":sum(np.prod(v.shape) for v in model.non_trainable_weights),"training_time_seconds":elapsed,"epochs_completed":len(h.history["loss"])}]).to_csv(OUT/"model_complexity.csv",index=False)
json.dump({"dataset_path":str(DATASET),"number_of_classes":NUM_CLASSES,"image_size":IMAGE_SIZE,"batch_size":BATCH_SIZE,"epochs":EPOCHS,"learning_rate":LEARNING_RATE,"optimizer":"Adam","loss":"SparseCategoricalCrossentropy","dropout":.5,"random_seed":RANDOM_SEED,"split_ratio":[.70,.15,.15],"use_class_weights":USE_CLASS_WEIGHTS},open(OUT/"experiment_config.json","w"),indent=2)
